# Web APIs and a released service

After this block you can request data from a web API with `httpx` and handle its failures, and you can turn a fitted model into a tested web API: saved with its versions, served with FastAPI and pydantic, packaged in a container and released by a CI/CD workflow. This serves the first learning outcome of Session 16: *request data from a web API and serve a model as a tested web API in a container, released with CI/CD*.

The block has two halves. In the first we are the **client** of someone else's API: Open-Meteo, which serves Berlin's daily weather. In the second we become the **server**: the tariff heading model of the case study (a description of goods goes in, the three most likely four-digit HS headings come out) turns into a service of its own. The complete service lives in [`../workspace/`](../workspace/README.md), with its tests, Dockerfile and workflows; this notebook builds its core step by step.

## Teaching map

The **core path** for the 45-minute lecture is marked with ●. Subsections marked *(depth: self-study or if time allows)* go further and can be skipped in class.

| Section | Minutes | What it delivers |
|---|---|---|
| ● How web APIs work | 9 | request and response, status codes, JSON, keys, pagination, rate limits, timeouts and retries |
| ● Requesting data with httpx | 9 | a live Open-Meteo request that fails gracefully, a 400 error, a mock server, the `WeatherClient` |
| ● From notebook to service | 8 | the fitted pipeline saved with joblib and skops, a version and a checksum, a lock file |
| ● A prediction API with FastAPI and pydantic, and its tests | 11 | `/health` and `/predict`, validation with 422, tests with `TestClient` |
| ● Containers with Docker | 5 | the workspace Dockerfile read layer by layer |
| ● Continuous delivery with GitHub Actions | 3 | CI on every push, a release on a version tag |
| Worked example · the service on 2024 requests | if time allows | the service answers real requests in five languages |
| **Total core** | **45** | |

```mermaid
flowchart LR
    W["Open-Meteo API"] -->|"httpx client"| N["Notebook"]
    N --> A["Artefact:<br/>model + metadata + uv.lock"]
    A --> S["Service:<br/>FastAPI + pydantic"]
    S --> T["Tests:<br/>TestClient, pytest"]
    T --> C["Container:<br/>Docker image"]
    C --> R["Release:<br/>GitHub Actions"]
    R --> U["Users:<br/>programs, dashboard"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## How web APIs work

### Why analysts need APIs

Some data arrive as files: the Inside Airbnb snapshot is a set of CSV files, published every quarter. Other data change every day or every minute, depend on what you ask for, or are too large to download as a whole. Their providers do not publish files; they answer questions. Weather is a typical case: Open-Meteo holds hourly values for every point on Earth since 1940, and nobody wants that as one file. You ask for exactly the days, the place and the variables you need.

Without an API, an analyst copies numbers by hand from a website: slow, error-prone and impossible to repeat next month. With an API, the same small program fetches tomorrow's data tomorrow, and the analysis can be rerun at any time.

In practice:

- **Open-Meteo** (open source, operated from Switzerland) serves weather forecasts and history as a JSON API without a key for non-commercial use; the German Weather Service (DWD) publishes its open data in a similar way.
- The **Python Package Index** has a JSON API (`https://pypi.org/pypi/<name>/json`) that tools such as uv use to find package versions.
- Nearly every business application (CRM, ticketing, payment) has a REST API that analysts use to pull data into reports, and open-data portals such as GovData run the catalogue software CKAN, whose API lists their datasets.

### What an API is

An **API** (application programming interface) is a defined way for one program to use another: which requests are possible, what they must contain and what comes back. You already use APIs: `pd.read_parquet` is part of the pandas API. A **web API** is an API used over the network with **HTTP**, the protocol of the web.

Think of a restaurant. You (the **client**) do not walk into the kitchen (the **server**). You order from a menu (the API documentation), and the waiter brings back exactly what you ordered, or tells you that the dish is not available.

- Each operation has an **endpoint**, a URL such as `https://archive-api.open-meteo.com/v1/archive` (past days) or `https://api.open-meteo.com/v1/forecast` (the next days).
- Most data APIs follow the **REST** style: URLs name resources (`/listings/42`), HTTP methods name actions (`GET` reads, `POST` submits or creates), and answers are usually **JSON**.
- An API is a **contract**: the provider documents it, and clients rely on it not changing without notice. Breaking changes come with a new **version** in the URL (`/v1/`, `/v2/`).

The request for the first three days of July 2024 in Berlin is one URL. Everything after `?` are **query parameters**, pairs of name and value joined by `&`:

```
https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41
  &start_date=2024-07-01&end_date=2024-07-03
  &daily=temperature_2m_mean,precipitation_sum,sunshine_duration&timezone=Europe%2FBerlin
```

`%2F` is the URL encoding of `/`: some characters may not appear literally in a URL. httpx encodes them for us.

> **Note.** Not every website is an API. Pages written for people return HTML and may block scripts; a documented API returns structured data and states its terms of use. Collecting data from pages written for people (*scraping*) needs care with terms of use and personal data.

### Requests and responses

An **HTTP request** has

- a **method**: `GET` (read), `POST` (submit data), `PUT` (replace), `DELETE` (remove);
- a **URL**, with optional query parameters;
- **headers**, metadata such as `User-Agent` (who is asking), `Accept: application/json` (what format we want) or `Authorization` (credentials);
- for `POST` and `PUT`, a **body**, usually JSON.

An **HTTP response** has a **status code**, headers (for example `content-type`) and a body. The first digit of the status tells who is to blame: 2 means success, 4 means the client made a mistake, 5 means the server did.

| Status | Meaning | What the client does |
|---|---|---|
| 200 OK | success | parse the body |
| 201 Created | a resource was created | read its id |
| 400 Bad Request, 422 Unprocessable Content | the request is invalid (Open-Meteo: a latitude of 152.52) | read the reason, fix the request; do not retry |
| 401 Unauthorized, 403 Forbidden | missing or insufficient credentials | check the key |
| 404 Not Found | wrong URL or no such resource | check the endpoint |
| 429 Too Many Requests | rate limit reached | wait (header `Retry-After`), then retry |
| 500, 502, 503 | server error | retry later, a limited number of times |

### JSON

**JSON** (JavaScript Object Notation) is the text format of most APIs. It has objects `{"key": value}`, arrays `[...]`, strings, numbers, `true` and `false`, and `null`. Python's `json` module turns them into `dict`, `list`, `str`, `int` or `float`, `bool` and `None`.

Open-Meteo returns a table **by column**: one array of dates and one array per variable, all of the same length. Next we parse a shortened answer and turn the columns into rows. `json.loads` turns text into Python objects; `zip(..., strict=True)` walks through the columns in step and raises an error if one is shorter than the others.

In [2]:
import json

text = '''{"latitude": 52.548, "longitude": 13.408, "timezone": "Europe/Berlin",
           "daily_units": {"time": "iso8601", "temperature_2m_mean": "°C", "sunshine_duration": "s"},
           "daily": {"time": ["2024-07-01", "2024-07-02", "2024-07-03"],
                     "temperature_2m_mean": [17.6, 15.1, 15.1],
                     "sunshine_duration": [33378.58, 20362.5, 32384.95]}}'''
payload = json.loads(text)
daily = payload["daily"]
rows = [{"date": d, "temperature": t, "sunshine_hours": round(s / 3600, 1)}
        for d, t, s in zip(daily["time"], daily["temperature_2m_mean"], daily["sunshine_duration"], strict=True)]
print(type(payload).__name__, payload["daily_units"]["sunshine_duration"])
rows[0]

dict s


{'date': '2024-07-01', 'temperature': 17.6, 'sunshine_hours': 9.3}

Errors come as JSON too. Open-Meteo answers a wrong latitude with status 400 and the body below. **Predict before running:** what does `error["error"]` print, a string or a Boolean?

In [3]:
error = json.loads('{"error": true, "reason": "Latitude must be in range of -90 to 90°. Given: 152.52."}')
print(repr(error["error"]), "|", error["reason"])

True | Latitude must be in range of -90 to 90°. Given: 152.52.


### Authentication

**Authentication** proves who is asking, so that the provider can bill, limit or refuse. Common forms:

- an **API key** sent in a header (`X-API-Key: <key>`) or as a query parameter (`&apikey=<key>`);
- a **bearer token** obtained through a login flow (OAuth 2.0) and sent as `Authorization: Bearer <token>`; tokens usually expire after minutes or hours.

Open-Meteo needs no key for non-commercial use; commercial customers send their requests to `customer-api.open-meteo.com` and add `&apikey=<key>`. Keys are secrets: they belong in **environment variables** or a `.env` file listed in `.gitignore`, never in code, notebooks or Git. Automated scanners search public repositories for leaked keys, and a key that was committed once must be revoked, even if the commit is deleted later.

**Predict before running.** `os.environ.get` returns `None` when a variable is not set. Which keys will `params` have on your machine?

In [4]:
key = os.environ.get("OPEN_METEO_API_KEY")       # never write the key itself into a notebook
params = {"latitude": 52.52, "longitude": 13.41}
if key:
    params["apikey"] = key                       # commercial customers only
sorted(params)

['latitude', 'longitude']

### Pagination

**Pagination** splits a long result into pages, so that one answer stays small. Three styles are common:

| Style | The client sends | Example |
|---|---|---|
| offset | `offset=40&limit=20` | rows 41 to 60 |
| page number | `page=3&per_page=20` | the third page |
| cursor | `cursor=<token from the last answer>` | "the next page after this one" |

Cursors are robust when new rows arrive while you page: an offset would shift and repeat or skip rows. The client loops until the answer has no next cursor. Next we play a paginated API with a Python function and page through it. `httpx.MockTransport(handler)` replaces the network with that function: it receives the `httpx.Request` and returns an `httpx.Response` (more on this below).

In [5]:
import httpx

DECISIONS = [f"BTI-{i:03d}" for i in range(1, 8)]          # seven records on the "server"

def paged_api(request: httpx.Request) -> httpx.Response:
    start = int(request.url.params.get("cursor", 0))
    page = DECISIONS[start:start + 3]                         # three per page
    nxt = start + 3 if start + 3 < len(DECISIONS) else None
    return httpx.Response(200, json={"items": page, "next_cursor": nxt})

api = httpx.Client(transport=httpx.MockTransport(paged_api), base_url="https://example.org")
items, cursor = [], 0
while cursor is not None:
    body = api.get("/decisions", params={"cursor": cursor}).json()
    items += body["items"]
    cursor = body["next_cursor"]
print(len(items), items[-1])

7 BTI-007


Open-Meteo has no pages, but the same idea applies to time: a client splits ten years into ten requests of one year each, so that each answer is small, progress is visible and a failure costs one year, not ten. A worked example by hand: the period from 1 November 2023 to 28 February 2025 splits into 1 November to 31 December 2023 (61 days), all of 2024 (366 days, a leap year) and 1 January to 28 February 2025 (59 days). The helper `yearly_chunks` in [weather.py](weather.py) does this split; next we check the hand calculation with it. `(end - start).days + 1` counts the days of a chunk with both ends included.

In [6]:
import sys
from datetime import date

sys.path.insert(0, "sessions/16-deployment-and-monitoring/workbooks")   # the folder of weather.py
from weather import yearly_chunks

for start, end in yearly_chunks(date(2023, 11, 1), date(2025, 2, 28)):
    print(start, end, (end - start).days + 1, "days")

2023-11-01 2023-12-31 61 days
2024-01-01 2024-12-31 366 days
2025-01-01 2025-02-28 59 days


### Rate limits, timeouts and retries

A **rate limit** caps the number of requests per time window, so that one client cannot overload a shared service. The free Open-Meteo API allows fewer than 600 calls per minute, 5,000 per hour and 10,000 per day, and counts a request for more than two weeks of data as several calls, about *days / 14* (and more for more than ten variables; Open-Meteo terms, checked October 2026). If a limit is exceeded, a server answers `429` and often sends a `Retry-After` header with the seconds to wait. Other examples: the GitHub REST API allows 60 requests per hour without authentication and 5,000 with a token, and reports the remaining quota in the headers `x-ratelimit-remaining` and `x-ratelimit-reset` (GitHub documentation, checked October 2026).

Next we count the calls Open-Meteo charges for a week, a year and the 3,830 days of the course's weather file. `max(1.0, days / 14)` encodes "at least one call".

In [7]:
def api_calls(days: int, n_variables: int = 3) -> float:
    # calls counted by Open-Meteo for one request (fractional counting)
    return max(1.0, days / 14) * max(1.0, n_variables / 10)

print(round(api_calls(7), 1), round(api_calls(366), 1), round(api_calls(3830), 1))

1.0 26.1 273.6


A **timeout** limits how long the client waits for an answer. Without one, a program can hang forever on a server that never responds. httpx waits 5 seconds by default; set the value explicitly. A **retry** repeats a request that failed for a reason that may pass (`429`, `5xx`, a timeout), with a maximum number of attempts and a growing wait: **exponential backoff** waits 1, 2, 4 seconds, so that a struggling server gets room to recover.

```mermaid
sequenceDiagram
  participant C as WeatherClient (httpx)
  participant S as Open-Meteo archive
  C->>S: GET archive?start_date=2016-01-01&end_date=2016-12-31
  S-->>C: 200 {"daily": {"time": [366 dates], ...}}
  C->>S: GET archive?start_date=2017-01-01&end_date=2017-12-31
  S-->>C: 429 Too Many Requests
  Note over C: wait (Retry-After or 1 s, then 2 s, ...)
  C->>S: GET ... 2017 (retry)
  S-->>C: 200 {"daily": {... 365 dates}}
  C->>S: GET archive?latitude=152.52&...
  S-->>C: 400 {"error": true, "reason": "Latitude must be in range ..."}
  Note over C: raise APIError with the reason, no retry
```

> **Note.** Retry only what can succeed later. Retrying `400` or `404` in a loop only adds load: the request will be just as wrong the next time. And always check the status before parsing: `response.json()` on an HTML error page raises an exception that hides the real cause.

## Requesting data with httpx

**httpx** is a Python library for HTTP. `httpx.get(url, params=..., headers=..., timeout=...)` sends one request and builds the part after `?` from the dictionary `params`. The response object has `status_code`, `headers`, `url` and `json()`, and `raise_for_status()` raises `httpx.HTTPStatusError` for any `4xx` or `5xx` status.

A request over the network can fail for reasons outside our control: no internet, a slow server, a rate limit. Code that runs in class, in a scheduled job or in CI therefore catches `httpx.HTTPError`, the base class of all httpx errors, and does something sensible instead of crashing.

Next we ask Open-Meteo's archive for the first week of July 2024 in Berlin. If the request fails, the cell falls back to a saved copy of the answer, so the rest of the notebook still runs.

In [8]:
ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
query = {"latitude": 52.52, "longitude": 13.41, "start_date": "2024-07-01", "end_date": "2024-07-07",
         "daily": "temperature_2m_mean,precipitation_sum", "timezone": "Europe/Berlin"}
try:
    r = httpx.get(ARCHIVE, params=query, timeout=10, headers={"User-Agent": "htw-course-example/0.1"})
    r.raise_for_status()
    week = r.json()["daily"]
    print("live:", r.status_code, r.headers["content-type"])
except httpx.HTTPError as err:                   # no network, timeout, 4xx or 5xx
    print("API not reachable, using the saved answer:", type(err).__name__)
    week = {"time": ["2024-07-01", "2024-07-02"], "temperature_2m_mean": [17.6, 15.1],
            "precipitation_sum": [7.9, 0.6]}
week["temperature_2m_mean"]

live: 200 application/json; charset=utf-8


[17.6, 15.1, 15.1, 15.5, 16.3, 21.5, 18.7]

A wrong request is answered with **400** and a JSON body that names the reason. Next we send a latitude of 152.52 on purpose. Two `except` clauses separate the cases: `httpx.HTTPStatusError` means the server answered with an error status (and `err.response` holds that answer); any other `httpx.HTTPError` means it did not answer at all.

In [9]:
try:
    bad = httpx.get(ARCHIVE, params={**query, "latitude": 152.52}, timeout=10)
    bad.raise_for_status()
except httpx.HTTPStatusError as err:
    print(err.response.status_code, err.response.json()["reason"])
except httpx.HTTPError as err:
    print("API not reachable:", type(err).__name__)

400 Latitude must be in range of -90 to 90°. Given: 152.52.


### One client for many requests *(depth: self-study or if time allows)*

`httpx.get` opens and closes a connection for every request. An `httpx.Client` keeps settings (headers, timeout, a base URL) in one place and **reuses connections**, which saves the handshake for every further request to the same server. Used in a `with` block, it closes its connections at the end. Next we send three requests through one client and look at `response.elapsed`, the time until the answer arrived.

In [10]:
try:
    with httpx.Client(base_url="https://archive-api.open-meteo.com", timeout=10,
                      headers={"User-Agent": "htw-course-example/0.1"}) as http:
        for month in ["01", "04", "07"]:
            resp = http.get("/v1/archive", params={**query, "start_date": f"2024-{month}-01",
                                                   "end_date": f"2024-{month}-07"})
            print(month, resp.status_code, f"{resp.elapsed.total_seconds() * 1000:.0f} ms",
                  resp.json()["daily"]["temperature_2m_mean"][:3])
except httpx.HTTPError as err:
    print("API not reachable:", type(err).__name__)

01 200 150 ms [5.3, 4.4, 8.7]
04 200 44 ms [12.8, 10.7, 9.5]


07 200 47 ms [17.6, 15.1, 15.1]


### Testing without the network

A test must not fail because someone else's server is down, slow or rate-limited. httpx lets us replace the network: a **transport** decides how requests are sent, and `httpx.MockTransport(handler)` sends them to a Python function that plays the server. Tests then run offline, fast and deterministically, and can simulate errors (`400`, `429`, `503`, a timeout) that are hard to provoke on a real server.

**Predict before running.** The fake server below answers `429` on the first call and data on the second. How many calls does `handler.calls` count at the end?

In [11]:
def handler(request: httpx.Request) -> httpx.Response:
    handler.calls += 1
    if handler.calls == 1:
        return httpx.Response(429, headers={"Retry-After": "0"})
    return httpx.Response(200, json={"daily": {"time": [request.url.params["start_date"]]}})

handler.calls = 0
fake = httpx.Client(transport=httpx.MockTransport(handler))
for attempt in range(3):                         # at most three attempts
    resp = fake.get(ARCHIVE, params={"start_date": "2024-07-01"})
    if resp.status_code != 429:                  # in real code: wait Retry-After seconds first
        break
print(resp.json(), handler.calls)

{'daily': {'time': ['2024-07-01']}} 2


### A client class: `WeatherClient`

A small client class keeps the details of one API in one place; the analysis code only calls `client.daily(start, end)`. The helper module [weather.py](weather.py) next to this notebook does this for Open-Meteo. `WeatherClient` *has* an `httpx.Client` (composition, Session 2) and adds what is specific to Open-Meteo: building the parameters, retrying after `429`, `5xx` and timeouts with exponential backoff, raising `APIError` with the server's reason for a bad request, and parsing each day into a validated `DailyWeather` record. Days with a missing value (`null`, which happens for the most recent days of the archive) are skipped and counted.

```mermaid
classDiagram
  class WeatherClient {
    +float latitude
    +float longitude
    +int max_retries
    -httpx.Client _http
    +daily(start, end) DailySeries
    +forecast(days) DailySeries
    +history(start, end) DailySeries
  }
  class DailySeries {
    +list~DailyWeather~ records
    +int skipped
  }
  class DailyWeather {
    +date day
    +float temperature
    +float precipitation
    +float sunshine_hours
  }
  WeatherClient *-- "1" Client : has
  WeatherClient ..> DailySeries : returns
  DailySeries o-- DailyWeather
  class Client["httpx.Client"]
```

Next we use it for July 2024 and for the forecast of the coming week. `len(series)` counts the valid days and `series.skipped` the days with a missing value.

In [12]:
from weather import APIError, DailyWeather, WeatherClient, parse_daily

try:
    with WeatherClient() as client:              # Berlin by default, no key needed
        july = client.daily(date(2024, 7, 1), date(2024, 7, 31))
        week_ahead = client.forecast(days=7)
    print(len(july), "days,", july.skipped, "skipped; first:", july.records[0])
    print("forecast from", week_ahead.records[0].day, "to", week_ahead.records[-1].day)
except (APIError, httpx.HTTPError) as err:
    print("API not reachable:", err)

31 days, 0 skipped; first: DailyWeather(day=datetime.date(2024, 7, 1), temperature=17.6, precipitation=7.9, sunshine_hours=9.271827777777778)
forecast from 2026-10-14 to 2026-10-20


The same client turns a wrong request into an `APIError` that carries the server's reason and the status code, and does not retry it:

In [13]:
try:
    with WeatherClient(latitude=152.52) as broken:
        broken.daily(date(2024, 7, 1), date(2024, 7, 7))
except APIError as err:
    print(type(err).__name__, err.status_code, "|", err)
except httpx.HTTPError as err:
    print("API not reachable:", type(err).__name__)

APIError 400 | HTTP 400: Latitude must be in range of -90 to 90°. Given: 152.52.


### Ten years, one year per request *(depth: self-study or if time allows)*

`WeatherClient.history(start, end)` pages by calendar year with `yearly_chunks`. Next we fetch the period of the course data, 1 January 2016 to 26 June 2026: eleven requests, about 274 calls by Open-Meteo's counting, far below the daily limit. If the network fails, we fall back to the stored file that the preparation script fetched once from the same API.

In [14]:
import pandas as pd

stored = pd.read_parquet("case-study/data/airbnb/weather_daily.parquet")
try:
    with WeatherClient() as client:
        history = client.history(date(2016, 1, 1), date(2026, 6, 26))
    daily_df = pd.DataFrame([d.to_dict() for d in history.records])
    daily_df["date"] = pd.to_datetime(daily_df["date"])
    print("live:", len(history), "days,", history.skipped, "skipped")
except (APIError, httpx.HTTPError) as err:
    print("API not reachable, using the stored file:", err)
    daily_df = stored.rename(columns={"temperature_2m_mean": "temperature", "precipitation_sum": "precipitation"})
daily_df.head(3)

live: 3830 days, 0 skipped


,date,temperature,precipitation,sunshine_hours
0,2016-01-01,1.8,0.4,3.14
1,2016-01-02,-2.4,0.0,2.50
2,2016-01-03,-8.0,0.0,6.96


**Validated records.** `DailyWeather` is a dataclass whose `__post_init__` checks every value: the temperature must lie between −40 and 45 °C, rain must not be negative, sunshine must lie between 0 and 24 hours. A record that breaks a rule is rejected with a clear message instead of entering the analysis. Next we try a temperature of 61 °C.

In [15]:
try:
    DailyWeather("2024-07-01", temperature=61.0, precipitation=0.0, sunshine_hours=10.0)
except ValueError as err:
    print("rejected:", err)

rejected: temperature: 61.0 lies outside -40 to 45


**Same data as the course file?** The archive is a *reanalysis* that the provider occasionally recomputes, so values can change slightly after the fact. Next we compare the live download with the stored file, day by day. `merge(on="date")` keeps the days present in both.

In [16]:
both = daily_df.merge(stored, on="date")
print(len(both), "days in both")
if "temperature_2m_mean" in both:
    print("largest temperature difference:", round((both["temperature"] - both["temperature_2m_mean"]).abs().max(), 2))

3830 days in both
largest temperature difference: 0.1


Record the download date whenever you use API data in a report. And request what you need once, store it, and read the stored file in later runs: that is polite to a free service and keeps the analysis working when the network is not. Session 12 reads `weather_daily.parquet` for exactly this reason.

### Testing the parsing offline *(depth: self-study or if time allows)*

Next a fake server answers with one complete and one incomplete day, as the archive does for the most recent days. The client should keep the first and count the second as skipped. `assert` stops with an error if the expectation is wrong; this is what a pytest function does.

In [17]:
def fake_archive(request: httpx.Request) -> httpx.Response:
    assert request.url.params["start_date"] == "2026-06-25"
    return httpx.Response(200, json={"daily": {
        "time": ["2026-06-25", "2026-06-26"],
        "temperature_2m_mean": [28.6, None],                  # the last day is not yet available
        "precipitation_sum": [0.0, None],
        "sunshine_duration": [57600.0, None]}})

with WeatherClient(transport=httpx.MockTransport(fake_archive)) as offline:
    series = offline.daily(date(2026, 6, 25), date(2026, 6, 26))
assert series.skipped == 1
print(len(series), "valid,", series.skipped, "skipped:", series.records[0])

1 valid, 1 skipped: DailyWeather(day=datetime.date(2026, 6, 25), temperature=28.6, precipitation=0.0, sunshine_hours=16.0)


`parse_daily` can also be tested without any client, because it takes a plain dictionary. Separating "fetch" from "parse" in this way makes both easy to test.

In [18]:
parse_daily({"daily": {"time": ["2024-07-01"], "temperature_2m_mean": [17.6],
                       "precipitation_sum": [7.9], "sunshine_duration": [33378.58]}})

DailySeries(records=[DailyWeather(day=datetime.date(2024, 7, 1), temperature=17.6, precipitation=7.9, sunshine_hours=9.271827777777778)], skipped=0)

### Nested JSON: live departures in Berlin *(depth: self-study or if time allows)*

Many APIs return **nested JSON**: objects inside objects. The community-run API `v6.vbb.transport.rest` returns the departures of Berlin's public transport; each departure contains a line, a stop and a delay in seconds. It is run by volunteers and allows 100 requests per minute per IP address (its documentation, checked October 2026), so use it sparingly. Stop `900100003` is S+U Alexanderplatz. Next we read the fields we need with chained keys such as `d["line"]["name"]`.

In [19]:
try:
    answer = httpx.get("https://v6.vbb.transport.rest/stops/900100003/departures",
                       params={"duration": 10, "results": 5}, timeout=5)
    answer.raise_for_status()
    print(pd.DataFrame([{"line": d["line"]["name"], "direction": d["direction"],
                         "delay_min": (d["delay"] or 0) / 60} for d in answer.json()["departures"]]))
except (httpx.HTTPError, KeyError, ValueError) as err:   # community service: may be unavailable
    print("VBB API not available:", type(err).__name__)

  line                   direction  delay_min
0   S9      S Spandau Bhf (Berlin)        0.0
1   M4                  Falkenberg        0.0
2   U2         Zoologischer Garten        0.0
3   S7  S Ahrensfelde Bhf (Berlin)        1.0
4  100          Mitte, Memhardstr.       14.0


In practice, httpx is also the HTTP layer of the OpenAI and Anthropic Python SDKs (Sessions 14 and 15), so the same ideas of timeouts, retries and transports apply when calling language models.

## From notebook to service: saving pipelines, versioning, pinned dependencies

A model in a notebook helps only the person who runs the notebook. To reach users it must become a **service**: a program that runs without us, loads the model from a file and answers requests. Most failures after deployment are not statistical but practical: a missing preprocessing step, a different library version, a hard-coded path. This section removes these causes one by one.

### Saving the whole pipeline

A **fitted pipeline** holds everything learned from the training data: for the case study, the TF-IDF vocabulary with its idf weights and the coefficients of the linear SVM, one row of coefficients per heading. **Serialisation** writes this Python object to a file; **deserialisation** restores it in another process, for example the API.

Save the **whole pipeline**, not only the classifier. Otherwise the service has to rebuild the preprocessing, and any small difference between training and serving (*training–serving skew*: another tokenisation, a forgotten lower-casing) silently degrades every prediction. One preprocessing function, used in both places, avoids it.

Next we train version 1 of the case-study model: word TF-IDF and a linear SVM (Session 13) on the 50,000-decision training sample, validated on the newest 20 % by date. `make_pipeline` chains the steps into one object. This takes about 15 seconds.

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline

def decision_text(description):                  # ONE preprocessing function for training and serving
    return " ".join((description or "").split())

df = pd.read_parquet("case-study/data/train_sample.parquet").sort_values("start_date", kind="stable")
X, y = [decision_text(t) for t in df["description"]], df["heading"].tolist()
cut = int(len(X) * 0.8)                          # time-based split: the newest 20 % validate
pipe = make_pipeline(TfidfVectorizer(sublinear_tf=True, min_df=2),
                     SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0))
pipe.fit(X[:cut], y[:cut])
acc = (pipe.predict(X[cut:]) == pd.Series(y[cut:])).mean()
print(f"validation accuracy {acc:.3f} on decisions from {df['start_date'].iloc[cut].date()}")

validation accuracy 0.778 on decisions from 2022-06-30


Next we save the pipeline with `joblib.dump`, which writes Python's **pickle** format efficiently for objects with large NumPy arrays, and load it again with `joblib.load`. `compress=3` trades a little time for a smaller file. We write into a temporary folder from `tempfile.mkdtemp()`; in the workspace the folder is `models/`.

In [21]:
import tempfile

import joblib

out = Path(tempfile.mkdtemp())
joblib.dump(pipe, out / "model.joblib", compress=3)
restored = joblib.load(out / "model.joblib")     # e.g. at start-up of the API process
examples = ["Spielzeug für Kinder ab 3 Jahren: Spielzeugauto aus Kunststoff mit Rädern, ohne Motor, zum Schieben",
            "Masque de protection respiratoire à usage unique, en non-tissé de polypropylène, avec élastiques"]
print(f"{(out / 'model.joblib').stat().st_size / 1e6:.0f} MB;", restored.predict(examples))

32 MB; ['9503' '6307']


### Why a pickle from a stranger is dangerous *(depth: self-study or if time allows)*

Loading a pickle **runs code**: the file can tell Python which function to call to rebuild an object. A malicious file can call anything. Next we build a harmless example: the class defines `__reduce__`, the hook that pickle uses, and makes the loader call `print`. In a real attack it would be a command that deletes files or sends your keys away.

In [22]:
import pickle

class Surprise:
    def __reduce__(self):                        # "to rebuild me, call print(...)"
        return (print, ("this line ran while the file was loading",))

data = pickle.dumps(Surprise())
pickle.loads(data)                               # nothing asked for this print

this line ran while the file was loading


**skops** stores scikit-learn objects in a format that is inspected before loading: types that are not known to be safe must be trusted explicitly. `sio.dump` writes the file, `sio.get_untrusted_types` lists the types in it that skops does not trust by default, and `sio.load(..., trusted=...)` loads it once you have read that list.

In [23]:
import skops.io as sio

sio.dump(pipe, out / "model.skops")
untrusted = sio.get_untrusted_types(file=out / "model.skops")
print("untrusted types:", untrusted)             # [] here; otherwise read the list before trusting it
safe = sio.load(out / "model.skops", trusted=untrusted)
safe.predict(examples[:1])

untrusted types: []


array(['9503'], dtype='<U4')

> **Note.** Never load a joblib or pickle file that you did not create or whose origin you cannot verify. Use skops, or a checksum of a file you made yourself (below). The Hugging Face Hub scans uploaded pickle files and introduced the `safetensors` format for the same reason.

### Versioning

A **version** identifies one artefact: the model file together with a small **metadata** file (version, library versions, training data, validation score, checksum). **Semantic versioning** writes it as MAJOR.MINOR.PATCH: a new major version for a change that users notice in behaviour (new training data or method), a minor version for a compatible addition (a new endpoint), a patch for a bug fix. Three things are versioned together:

| What | How | Where |
|---|---|---|
| code | Git commits and tags (`v1.0.0`) | GitHub |
| model artefact | version in `metadata.json`, file checksum | GitHub release or a model registry |
| data | file name, date range and checksum of the training data | `metadata.json`; DVC or a data catalogue for larger setups |

A **checksum** (here SHA-256) is a fingerprint of a file: change one byte and it changes completely. The service compares the checksum of the model file with the one in the metadata and refuses a file that is not the released one. Next we build the metadata. `hashlib.sha256(bytes).hexdigest()` computes the checksum.

In [24]:
import hashlib

import sklearn

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

meta = {"model_version": "1.0.0", "scikit_learn": sklearn.__version__,
        "training_data": {"file": "train_sample.parquet", "sha256": sha256("case-study/data/train_sample.parquet")[:12],
                          "date_max": str(df["start_date"].iloc[cut - 1].date())},
        "validation": {"scheme": "time-based, newest 20 %", "accuracy": round(float(acc), 3)},
        "model_sha256": sha256(out / "model.joblib")[:16]}
(out / "metadata.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=1))

{
 "model_version": "1.0.0",
 "scikit_learn": "1.9.1",
 "training_data": {
  "file": "train_sample.parquet",
  "sha256": "93362067d074",
  "date_max": "2022-06-29"
 },
 "validation": {
  "scheme": "time-based, newest 20 %",
  "accuracy": 0.778
 },
 "model_sha256": "7c32f841a6efa8a4"
}


When a user reports a strange prediction, or monitoring shows a drop in quality (Block 2), we must know which model made the prediction, on which data it was trained, and how to restore the previous one. Without versions, a rollback is guesswork. Model registries such as MLflow store each model file with its metrics, parameters and environment for the same purpose.

> **Note.** Keep the version in one place (the metadata) and let everything else read it: the API response, the container tag and the model card.

### Pinned dependencies

A model file is only valid together with the code that reads it. scikit-learn does not guarantee that a pickled model loads in another version, and records the version it was saved with. **Predict before running:** where does the fitted pipeline keep that version? The next cell reads it from the object's pickled state.

In [25]:
pickled_state = pipe.__getstate__()
print(pickled_state.get("_sklearn_version"), "| running:", sklearn.__version__)

1.9.1 | running: 1.9.1


**Pinning** means recording the exact version of every package, including the indirect ones. With uv:

- `pyproject.toml` lists the **direct** dependencies with lower bounds (`scikit-learn>=1.9`);
- `uv lock` resolves all direct and indirect dependencies into **`uv.lock`**, with exact versions and file hashes;
- `uv sync --frozen` recreates exactly that environment on another machine, in CI or in a container.

```bash
cd sessions/16-deployment-and-monitoring/workspace
uv add scikit-learn fastapi        # adds lower bounds to pyproject.toml and updates uv.lock
uv lock --upgrade-package fastapi  # a deliberate update of one package
uv sync --frozen                   # recreate exactly the locked versions (CI, Docker)
uv tree --depth 1                  # what is installed, and why
```

Next we compare the two files of the workspace: how many packages are named directly, and how many are locked. `tomllib` (standard library) reads TOML files.

In [26]:
import tomllib

ws = Path("sessions/16-deployment-and-monitoring/workspace")
project = tomllib.loads((ws / "pyproject.toml").read_text())
lock = tomllib.loads((ws / "uv.lock").read_text())
print(len(project["project"]["dependencies"]), "direct dependencies:", project["project"]["dependencies"][:3], "...")
print(len(lock["package"]), "packages pinned in uv.lock, e.g.",
      [(p["name"], p["version"]) for p in lock["package"] if p["name"] in ("scikit-learn", "fastapi")])

9 direct dependencies: ['fastapi>=0.143', 'pydantic>=2.14', 'uvicorn>=0.54'] ...
63 packages pinned in uv.lock, e.g. [('fastapi', '0.143.0'), ('scikit-learn', '1.9.1')]


Settings that differ between machines (paths, ports, keys) are not pinned in code but read from **environment variables**, for example `MODEL_DIR=models` on a laptop and `MODEL_DIR=/app/models` in the container. The *Twelve-Factor App* method, widely used for web services, asks for exactly this: dependencies declared explicitly (factor II) and configuration in the environment (factor III).

In [27]:
model_dir = Path(os.environ.get("MODEL_DIR", "models"))   # configuration from the environment
model_dir

PosixPath('models')

> 📖 **Read more:** [extras/01-skops-text-classification-model-card.ipynb](extras/01-skops-text-classification-model-card.ipynb) (skops documentation, MIT): trains a text classifier, saves it and writes its model card with skops; Block 2 uses the model-card part.

## A prediction API with FastAPI and pydantic, and its tests

Now we are on the server side. A **web API** of our own separates the model from the programs that use it: a case-management system or a dashboard needs only a URL and the JSON format, and the model can be replaced without changing them. An **endpoint** is one URL with one method:

- `GET /health` reports that the service is alive and which model it runs; load balancers and container platforms call it every few seconds;
- `POST /predict` receives a description of goods and returns the three highest-scoring headings.

**FastAPI** maps Python functions to endpoints with decorators such as `@app.get("/health")` or `@app.post("/predict")`. Function parameters become **path parameters** (`/headings/{heading}`), **query parameters** (`?k=3`) or the request **body**, and returned objects are sent as JSON. **uvicorn** is the server program that runs the app and listens for requests.

**pydantic** describes the request and the response as Python classes with typed fields. FastAPI checks every request against the request class *before* our function runs: an empty description or a missing field is rejected with status **422** and a message that names the field. This keeps malformed requests away from the model and gives clear errors instead of crashes. From the same classes FastAPI generates interactive documentation at `/docs`.

The model is loaded **once at start-up** (in the `lifespan` function), not per request: loading a 30 MB pipeline for every request would make each request slow and the service fragile under load.

```mermaid
sequenceDiagram
    participant C as Client (case system, dashboard)
    participant F as FastAPI
    participant P as pydantic
    participant M as Model
    C->>F: POST /predict {"description": "Stiefel ..."}
    F->>P: validate against DecisionIn
    alt invalid
        P-->>C: 422 with the failing field
    else valid
        F->>M: decision_function(decision_text(...))
        M-->>F: one score per heading
        F->>P: build PredictionOut (top k + heading texts)
        F-->>C: 200 {"heading": "6404", "top": [...], "model_version": "1.0.0"}
    end
```

Next we define the request and response models. `Field(min_length=1)` rejects an empty string, `str_strip_whitespace=True` strips spaces first (so `"   "` counts as empty), `pattern=r"^\d{4}$"` demands four digits, and `extra="forbid"` rejects fields we did not ask for, such as a client sending the answer `"heading"`.

In [28]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class DecisionIn(BaseModel):                     # the request body
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    description: str = Field(min_length=1, max_length=20_000)
    language: str | None = None

class HeadingScore(BaseModel):
    heading: str = Field(pattern=r"^\d{4}$")
    score: float                                 # margin of the linear SVM, not a probability
    heading_description: str

class PredictionOut(BaseModel):                  # the response body
    heading: str
    top: list[HeadingScore]
    model_version: str

### pydantic on its own *(depth: self-study or if time allows)*

pydantic works without FastAPI. `Model.model_validate(dict)` checks a dictionary and returns an object or raises `ValidationError`, whose `.errors()` lists every problem with its location and type. FastAPI turns exactly this list into the body of a 422 answer. **Predict before running:** how many errors does the second dictionary produce?

In [29]:
print(DecisionIn.model_validate({"description": "  Schuhe aus Leder  ", "language": "de"}))
try:
    DecisionIn.model_validate({"description": "", "heading": "6403"})
except ValidationError as err:
    for e in err.errors():
        print(e["loc"], e["type"], "|", e["msg"])

description='Schuhe aus Leder' language='de'
('description',) string_too_short | String should have at least 1 character
('heading',) extra_forbidden | Extra inputs are not permitted


Next comes the app, a compact version of `workspace/src/tariff_service/app.py`. `@asynccontextmanager` turns `lifespan` into the start-up and shut-down hook: the code before `yield` runs once when the service starts. `response_model=PredictionOut` makes FastAPI check and document the output. The query parameter `k: int = Query(3, ge=1, le=10)` lets a client ask for more suggestions, between 1 and 10. A linear SVM has no probabilities: the scores are its margins, above 0 the model votes for a heading, below 0 it is not convinced. `np.argsort(-scores)[:k]` gives the positions of the k largest scores.

In [30]:
from contextlib import asynccontextmanager

import numpy as np
from fastapi import FastAPI, Query

heading_texts = pd.read_parquet("case-study/data/nomenclature.parquet").set_index("heading")["heading_description"]
state = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    state["model"] = sio.load(out / "model.skops", trusted=untrusted)   # once, at start-up
    yield
    state.clear()                                                       # at shut-down

app = FastAPI(title="Tariff heading API", version=meta["model_version"], lifespan=lifespan)

@app.get("/health")
def health() -> dict:
    return {"status": "ok", "model_version": meta["model_version"]}

In [31]:
@app.post("/predict", response_model=PredictionOut)
def predict(decision: DecisionIn, k: int = Query(3, ge=1, le=10)) -> PredictionOut:
    model = state["model"]
    scores = model.decision_function([decision_text(decision.description)])[0]
    top = [HeadingScore(heading=model.classes_[i], score=round(float(scores[i]), 3),
                        heading_description=heading_texts.get(model.classes_[i], ""))
           for i in np.argsort(-scores)[:k]]
    return PredictionOut(heading=top[0].heading, top=top, model_version=meta["model_version"])

In the workspace, `MODEL_DIR=models uv run uvicorn tariff_service.app:app --reload` starts a real server, and the documentation appears at `http://localhost:8000/docs`. In a notebook we do not need a server: FastAPI's **`TestClient`** sends HTTP requests to the app inside the same Python process. `with TestClient(app) as client:` also runs the lifespan, as a real start would.

**Predict before running.** Which heading do you expect for a pair of women's boots with rubber soles and leather uppers? (Chapter 64 is footwear; 6403 has leather uppers, 6404 textile uppers.)

In [32]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    print(client.get("/health").json())
    r = client.post("/predict", json={"description": "Stiefel für Damen, mit Laufsohlen aus Kautschuk "
                                      "und Oberteil aus Leder, den Knöchel bedeckend", "language": "de"})
    print(r.status_code)
    for t in r.json()["top"]:
        print(t["heading"], t["score"], t["heading_description"][:90])

{'status': 'ok', 'model_version': '1.0.0'}
200
6404 -0.322 Footwear; with outer soles of rubber, plastics, leather or composition leather and uppers 
6403 -0.508 Footwear; with outer soles of rubber, plastics, leather or composition leather and uppers 
4203 -0.872 Articles of apparel and clothing accessories, of leather or of composition leather


The model puts 6404 first and the correct 6403 second, both with negative scores: it is not convinced. This is a finding for the evaluation and the model card, and a reason to show low-score answers as suggestions only and route them to an officer (Session 14, abstention).

Next we send three invalid requests: a description of spaces only, a wrong field name, and a client that tries to send the answer. Each should come back as 422 without reaching the model. `r.json()["detail"][0]` holds pydantic's explanation, and a `k` of 50 breaks the query rule.

In [33]:
with TestClient(app) as client:
    for bad in [{"description": "   "}, {"text": "Schuhe"}, {"description": "Schuhe", "heading": "6403"}]:
        r = client.post("/predict", json=bad)
        print(r.status_code, r.json()["detail"][0]["type"], r.json()["detail"][0]["loc"])
    r = client.post("/predict", params={"k": 50}, json={"description": "Schuhe"})
    print(r.status_code, r.json()["detail"][0]["msg"])

422 string_too_short ['body', 'description']
422 missing ['body', 'description']
422 extra_forbidden ['body', 'heading']
422 Input should be less than or equal to 10


### The generated documentation *(depth: self-study or if time allows)*

FastAPI describes the API in the **OpenAPI** format at `/openapi.json`; the page `/docs` renders it. Other teams generate client code from this file, and it is the same format that many public open-data portals publish. Next we list the documented endpoints and the fields of the request body.

In [34]:
with TestClient(app) as client:
    spec = client.get("/openapi.json").json()
print({path: list(ops) for path, ops in spec["paths"].items()})
print(spec["components"]["schemas"]["DecisionIn"]["required"], list(spec["components"]["schemas"]["DecisionIn"]["properties"]))

{'/health': ['get'], '/predict': ['post']}
['description'] ['description', 'language']


### Tests for the service

An **automated test** calls the code and **asserts** what the result must be; **pytest** (Session 2) runs every function whose name starts with `test_`. Three kinds of test fit a model service:

- **contract tests** check the interface: status codes, field names, three headings of four digits, scores in descending order;
- **validation tests** check that invalid input is rejected with 422;
- **behavioural tests** check the model on a few obvious cases that any acceptable model gets right, for example that a plain description of a plastic toy car gets heading 9503. They catch a broken model file or a swapped label order, not small changes in accuracy (Ribeiro et al. 2020 call these *minimum functionality tests*).

A borderline case like the boots above is not a test case but a question for the evaluation: a test that fails whenever a better model makes a different choice blocks every improvement. Next we write one test that covers all three kinds and call it ourselves; in the workspace, `uv run pytest -q` finds and runs such functions.

In [35]:
def test_predict_contract_validation_behaviour():
    with TestClient(app) as client:
        body = client.post("/predict", json={"description": examples[0]}).json()
        scores = [t["score"] for t in body["top"]]
        assert set(body) == {"heading", "top", "model_version"}            # contract
        assert len(scores) == 3 and scores == sorted(scores, reverse=True)  # contract
        assert client.post("/predict", json={"description": ""}).status_code == 422   # validation
        assert body["heading"] == "9503"                                    # behaviour

test_predict_contract_validation_behaviour()
print("test passed")

test passed


Tests must not depend on the real data or on a large model file. The workspace trains a **fixture model** on 16 built-in descriptions in a few milliseconds (`tests/conftest.py`), and runs its tests with:

```bash
cd sessions/16-deployment-and-monitoring/workspace
uv sync && uv run pytest -q
```

### How fast is one prediction? *(depth: self-study or if time allows)*

Capacity planning starts with the time per request. Next we send 50 requests through `TestClient` and divide. `time.perf_counter()` is a precise clock for measuring durations. The number includes FastAPI's validation and JSON handling, but no network.

In [36]:
import time

with TestClient(app) as client:
    t0 = time.perf_counter()
    for _ in range(50):
        client.post("/predict", json={"description": examples[1]})
    per_request = (time.perf_counter() - t0) / 50
print(f"{per_request * 1000:.1f} ms per request on this machine")

112.9 ms per request on this machine


In practice, Breck et al. (Google, 2017) list tests of the serving infrastructure and of model behaviour in the *ML Test Score*, a rubric for production readiness, and Ribeiro et al. (2020) used simple behavioural tests (*CheckList*) to find failures in commercial text-classification APIs.

> **Note.** Put the model version into every response and every log line. It costs nothing and makes every later analysis traceable (Block 2).

## Containers with Docker

"It works on my machine" happens because machines differ: another Python, other package versions, a missing system library, a file in a different place. A **container image** bundles the service with everything it needs to run: an operating-system layer, Python, the locked packages, the code and the model file. A **container** is a running instance of an image, and it behaves the same on a laptop, a university server or a cloud platform. Think of a shipping container: the crane and the ship do not care what is inside. **Docker** is the most widely used tool to build and run containers; **Podman** is an open-source alternative with the same commands.

A **Dockerfile** lists the steps to build the image:

| Instruction | Meaning |
|---|---|
| `FROM python:3.14-slim` | start from a published base image |
| `WORKDIR /app` | the folder in which the next steps run |
| `COPY src ./src` | copy files from the project into the image |
| `RUN uv sync --frozen` | run a command while building |
| `ENV MODEL_DIR=/app/models` | set an environment variable |
| `USER appuser` | run as this (non-root) user |
| `EXPOSE 8000` | document the port the service listens on |
| `HEALTHCHECK ...` | a command that tells the platform whether the container is healthy |
| `CMD ["uvicorn", ...]` | the command that starts the service |

Each instruction creates a **layer**, and Docker reuses unchanged layers from earlier builds (the **build cache**). Therefore copy `pyproject.toml` and `uv.lock` and install the dependencies *first*, then copy the code. A worked example: you change one line in `app.py` and rebuild. The layers up to the dependency install are reused; only the layers from `COPY src` onwards are rebuilt, in seconds instead of minutes.

The Dockerfile of the workspace (`workspace/Dockerfile`):

```dockerfile
FROM python:3.14-slim

# uv from its official image, pinned to a minor version
COPY --from=ghcr.io/astral-sh/uv:0.13 /uv /uvx /bin/
ENV UV_COMPILE_BYTECODE=1 UV_LINK_MODE=copy UV_PYTHON_DOWNLOADS=never
WORKDIR /app

# 1. dependencies first: this layer is reused until pyproject.toml or uv.lock change
COPY pyproject.toml uv.lock README.md ./
RUN uv sync --frozen --no-dev --no-install-project --no-cache

# 2. then the code (installed as a package, not editable) and the model files
COPY src ./src
RUN uv sync --frozen --no-dev --no-editable --no-cache
COPY models ./models
ENV MODEL_DIR=/app/models PATH="/app/.venv/bin:$PATH"

# 3. run as a non-root user; /app/data holds the SQLite prediction log (Block 2)
RUN useradd --create-home --uid 10001 appuser && mkdir -p /app/data && chown appuser /app/data
USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s --start-period=10s \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["uvicorn", "tariff_service.app:app", "--host", "0.0.0.0", "--port", "8000"]
```

A `.dockerignore` file keeps things out of the image that do not belong there: `.venv`, `.git`, tests, notebooks, data files and `.env`. The image of the workspace contains no tests, notebooks or data.

Build and run it where Docker or Podman is installed (shown, not run in this notebook):

```bash
cd sessions/16-deployment-and-monitoring/workspace
uv run python -m tariff_service.train --data ../../../case-study/data/train_sample.parquet --version 1.0.0
docker build -t tariff-service:1.0.0 .
docker run --rm -p 8000:8000 tariff-service:1.0.0     # then open http://localhost:8000/docs
docker images tariff-service                           # size of the image
```

`-p 8000:8000` connects port 8000 of your machine to port 8000 of the container. Any program can then use the service with the httpx code of the first half:

```python
# requires the running container
r = httpx.post("http://localhost:8000/predict",
               json={"description": "Masque de protection à usage unique, en non-tissé", "language": "fr"})
print(r.status_code, r.json()["heading"], [t["heading"] for t in r.json()["top"]])
```

In practice, cloud platforms such as Google Cloud Run, AWS App Runner and Azure Container Apps run any container image that listens on a port, and Kubernetes (released by Google in 2014, now maintained by the Cloud Native Computing Foundation) runs and restarts containers across many machines. Block 2 shows where the container runs.

> **Note.** Never copy secrets (API keys, passwords, `.env` files) into an image: anyone who can pull the image can read them. Pass them at run time as environment variables or through the platform's secret store. Use a slim base image, a `.dockerignore` and a non-root user.

## Continuous delivery with GitHub Actions

**Continuous integration** (CI, Session 2) tests every push and pull request automatically on a clean machine. **Continuous delivery** (CD) extends this: every change that passes the tests produces a releasable artefact, and a release is one deliberate action, here pushing a version tag such as `v1.0.0`. **Continuous deployment** goes one step further and deploys every passing change automatically.

In **GitHub Actions**, a **workflow** is a YAML file in `.github/workflows/` at the repository root. It names the **events** that trigger it (`push`, `pull_request`, a tag) and **jobs** that run on fresh virtual machines (`runs-on: ubuntu-latest`). A job executes **steps**: either a published **action** (`uses: actions/checkout@v7`) or a shell command (`run: uv run pytest -q`). A job can wait for another (`needs: test`).

The workspace has two workflows:

| Workflow | Trigger | Steps |
|---|---|---|
| `ci.yml` | every push to `main` and every pull request | ruff, pytest; then build the image with the fixture model and smoke-test `/health` and `/predict` |
| `cd.yml` | a tag `v*.*.*` | download the model files of that GitHub release; check that the version in `metadata.json` equals the tag; build and push the image to `ghcr.io` |

The model file is attached to the **GitHub release**, not committed to Git: model files are large binary files, and the training data may not be redistributed. A release as a Git history: features are merged through pull requests, and a tag on `main` triggers the release.

```mermaid
gitGraph
    commit id: "service v1 code"
    branch feature-batch
    commit id: "batch endpoint"
    commit id: "tests"
    checkout main
    merge feature-batch id: "PR merged (CI green)"
    commit id: "model card 1.0.0" tag: "v1.0.0"
    branch retrain-2024
    commit id: "2024 labels in training"
    checkout main
    merge retrain-2024 id: "PR merged"
    commit id: "model card 2.0.0" tag: "v2.0.0"
```

The release job of `workspace/.github/workflows/cd.yml`, shortened:

```yaml
name: cd
on:
  push:
    tags: ["v*.*.*"]
permissions:
  contents: read
  packages: write          # push to the GitHub container registry
jobs:
  release-image:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7
      - name: Download the model files of this release
        env:
          GH_TOKEN: ${{ github.token }}
        run: gh release download "${GITHUB_REF_NAME}" --pattern "model.joblib" --pattern "metadata.json" --dir models
      - uses: docker/login-action@v4
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - id: meta
        uses: docker/metadata-action@v6
        with:
          images: ghcr.io/${{ github.repository }}
          tags: type=semver,pattern={{version}}
      - uses: docker/build-push-action@v7
        with:
          context: .
          push: true
          tags: ${{ steps.meta.outputs.tags }}
```

The release itself, from the command line with the GitHub CLI:

```bash
uv run python -m tariff_service.train --data ../../../case-study/data/train_sample.parquet --version 1.0.0
git tag v1.0.0 && git push origin v1.0.0
gh release create v1.0.0 models/model.joblib models/metadata.json --title "Model 1.0.0" --notes-file MODEL_CARD.md
```

CD makes a release repeatable and auditable: the image for version 2.0.0 is built the same way as for 1.0.0, from a tagged commit, with a checked model file. Nobody builds an image on a laptop with uncommitted changes, and a rollback means redeploying the image of the previous tag.

Next we write the workflow's version check in Python: the tag must equal the version in the metadata, or the release stops. **Predict before running:** what happens with the tag `v1.1.0`?

In [37]:
for tag in ["v1.0.0", "v1.1.0"]:                 # in the workflow: os.environ["GITHUB_REF_NAME"]
    try:
        assert "v" + meta["model_version"] == tag, f"metadata {meta['model_version']} != tag {tag}"
        print("release", tag, "matches the model metadata")
    except AssertionError as err:
        print("release stopped:", err)

release v1.0.0 matches the model metadata
release stopped: metadata 1.0.0 != tag v1.1.0


In practice, most open-source Python projects, including pandas and scikit-learn, run their test suites on every pull request and publish releases from tags, and companies protect their main branch so that a pull request can only be merged when the CI checks pass (*branch protection*).

> **Note.** Workflows run only from `.github/workflows/` at the **root** of a repository. The files in `workspace/.github/` are examples; your team copies them to the root of its own repository. A workflow that pushes images needs credentials: use the built-in `GITHUB_TOKEN` with minimal `permissions`, store other keys as repository secrets, and never print them in a step.

## Worked example · Does the service answer real 2024 requests sensibly?

*If time allows.* The service was trained on decisions up to mid-2022 in this notebook. Next we send real requests from 2024 (`case-study/data/test.parquet`, the hidden-label test set) through the API, one per main language, and look at what comes back. `groupby("language").head(1)` takes the first request of each language.

In [38]:
test = pd.read_parquet("case-study/data/test.parquet")
sample = (test[test["start_date"].dt.year == 2024]
          .loc[lambda d: d["language"].isin(["de", "fr", "nl", "cs", "pl"])]
          .groupby("language").head(1))
rows = []
with TestClient(app) as client:
    for _, req in sample.iterrows():
        body = client.post("/predict", json={"description": req["description"], "language": req["language"]}).json()
        rows.append({"language": req["language"], "chars": len(req["description"]),
                     "heading": body["heading"], "score": body["top"][0]["score"],
                     "text": body["top"][0]["heading_description"][:45]})
pd.DataFrame(rows)

,language,chars,heading,score,text
0,de,666,7307,1.145,"Tube or pipe fittings (e.g. couplings, elbows"
1,fr,390,2309,0.669,Preparations of a kind used in animal feeding
2,nl,838,8501,-0.643,Electric motors and generators (excluding gen
3,cs,315,6307,0.768,Textiles; made up articles n.e.c. in chapter
4,pl,276,2933,-0.486,Heterocyclic compounds with nitrogen hetero-a


**Reading the result.** Look at the scores: a positive margin means the model votes for the heading, a negative one means it is unsure. The labels of these requests are released only in Block 2, so we cannot yet say which answers are right; Block 2 measures exactly that, per language, and finds large differences. A service would show low-score answers as suggestions only.

## Check your understanding

1. An API answers `429` with `Retry-After: 30`. What should the client do? And after a `400`?
2. Why do tests of an API client use `httpx.MockTransport` instead of the real server? Name two errors a mock can simulate that are hard to provoke on the real server.
3. Why do we save the whole pipeline and not only the classifier, and why is a joblib file from a stranger dangerous?
4. A request sends `{"description": "Stiefel aus Leder", "heading": "6403"}`. What does our API answer, and why is that desirable?
5. Why does the Dockerfile copy `uv.lock` and install the packages before it copies `src/`? What is the difference between continuous delivery and continuous deployment?

## Exercises (30 min)

Work in pairs. The lecturer picks which exercises to set; the others are good practice at home. Answer *Paper*, *Pairs* and *Predict* tasks in the markdown cell below the task, *Code* tasks in the empty code cells.

### 1.1 · Tomorrow's weather in JSON

*Paper · 5 min*

> The printout below is a shortened answer of a weather API for Berlin. Write the Python expression that returns tomorrow's maximum temperature, and one that finds the day with the most rain.
>
> ```json
> {
>   "latitude": 52.52, "longitude": 13.41, "timezone": "Europe/Berlin",
>   "daily_units": {"time": "iso8601", "temperature_2m_max": "°C", "precipitation_sum": "mm"},
>   "daily": {
>     "time": ["2026-10-12", "2026-10-13", "2026-10-14", "2026-10-15"],
>     "temperature_2m_max": [16.2, 14.8, 12.1, 13.5],
>     "precipitation_sum": [0.0, 3.4, 7.9, 0.2]
>   }
> }
> ```
>
> The answer has been parsed with `data = json.loads(text)`; today is the first date.

*Source:* original exercise for this course; background: [Open-Meteo API documentation](https://open-meteo.com/en/docs)

*Your answer:*

### 1.2 · API arithmetic

*Paper · 10 min*

> Your team needs the daily weather (three variables) of Berlin, Hamburg and Munich from 1 March 2022 to 31 August 2025. The client pages by calendar year, one request per city and year. Open-Meteo counts a request for more than 14 days as several calls: one request costs *days / 14* calls (at least 1). The free API allows 600 calls per minute, 5,000 per hour and 10,000 per day.
>
> 1. Which requests does the client send for one city (start and end date of each)? How many requests in total for the three cities?
> 2. How many calls does Open-Meteo count for the whole download?
> 3. A teammate puts the download at the top of the analysis notebook, which the team runs about 50 times a day. Which limit is reached first, and what should the notebook do instead?

*Source:* original exercise for this course; background: [Open-Meteo terms and pricing](https://open-meteo.com/en/pricing)

*Your answer:*

### 1.3 · Capacity in your head

*Paper · 5 min*

> The tariff service receives 100 requests per second, and one prediction takes 50 ms. How many workers (processes that each answer one request at a time) does the service need?

*Source:* [Little's law, Wikipedia](https://en.wikipedia.org/wiki/Little%27s_law)

*Your answer:*

### 1.4 · "Works on my machine"

*Pairs · 10 min*

> List five reasons why a notebook that runs on your laptop fails on someone else's computer. For each reason, name what in the workspace of this session prevents it.

*Source:* original exercise for this course; background: [The Twelve-Factor App](https://12factor.net/) (factors II Dependencies and III Config)

*Your answer:*

### 1.5 · The pickle mismatch

*Predict · 10 min*

> You save a fitted pipeline with `joblib.dump` using one version of scikit-learn and load it with another version. What happens? Write down your prediction first. Then try it and compare: save the pipeline in this notebook, and load it in a terminal with an older version, for example `uv run --no-project --with "scikit-learn==1.7.*" --with joblib python -c "import joblib; print(joblib.load('pipe.joblib'))"`.

*Source:* [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html)

*Your answer:*

### 1.6 · An endpoint for heading texts

*Code · 15 min*

> Customs officers want to look up the English text of a heading. Write a small FastAPI app with one endpoint, `GET /headings/{heading}`, that returns `{"heading": "6403", "text": "..."}` from the dictionary `HEADINGS` loaded below. A heading that does not exist (for example `0100`) is answered with 404, and a path that is not four digits (for example `64x3` or `640`) with 422. Then check all three cases with `TestClient`, without starting a server.
>
> Hint: `help(fastapi.Path)` shows how a path parameter gets a `pattern`.

*Source:* original exercise for this course; background: [FastAPI tutorial: path parameters and numeric validations](https://fastapi.tiangolo.com/tutorial/path-params-numeric-validations/)

In [39]:
nomenclature = pd.read_parquet("case-study/data/nomenclature.parquet")
HEADINGS = dict(zip(nomenclature["heading"], nomenclature["heading_description"]))
print(len(HEADINGS), "headings, for example 6403:", HEADINGS["6403"])

1229 headings, for example 6403: Footwear; with outer soles of rubber, plastics, leather or composition leather and uppers of leather


### 1.7 · A server that fails

*Code · 15 min*

*Stretch:*

> The `WeatherClient` in [weather.py](weather.py) retries after HTTP 429, a server error (5xx) or a timeout. Test with a mock server instead of Open-Meteo what it does when the server answers `503` twice and then returns one valid day. Use an `httpx.MockTransport` as the server, and pass the client a `sleep` function that records the waits instead of waiting. How many requests does the client send, and how long would it have waited in total? What happens if the server answers `503` five times in a row?
>
> Hint: `help(WeatherClient)` lists the parameters `transport`, `max_retries` and `sleep`.

*Source:* original exercise for this course; background: [HTTPX documentation: transports](https://www.python-httpx.org/advanced/transports/)

In [40]:
ONE_DAY = {"daily": {"time": ["2024-07-01"], "temperature_2m_mean": [17.6],
                     "precipitation_sum": [7.9], "sunshine_duration": [33378.58]}}

## Debrief (15 min)

- **1.1 and 1.2.** Compare two JSON expressions: one that indexes the columns by position (`data["daily"]["temperature_2m_max"][1]`) and one that looks for a list of day objects that does not exist. In the request count, look for a forgotten leap year (2024) and for 2022 counted from January.
- **1.3 and 1.4.** Ask one pair for its five reasons and map each to a file of the workspace: `uv.lock`, `MODEL_DIR` from the environment, the Dockerfile, the tests, the checksum in `metadata.json`.
- **1.5.** "No error" does not mean "correct": a warning is easy to overlook. That is why the versions are pinned and recorded.
- **1.6.** Look for `heading: int`. It turns `0101` into `101` and accepts `640`; the heading must be a string with the pattern `^\d{4}$`.
- **Bridge to Block 2.** The service now runs and is tested. Next: where it runs for real users (front end, back end and database in the cloud), and how we notice that the model ages once the requests of 2024 and later arrive.

## Further reading

- Mozilla. *MDN Web Docs: An overview of HTTP; HTTP response status codes*. [developer.mozilla.org/en-US/docs/Web/HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP)
- Encode and contributors. *HTTPX documentation: Quickstart; Timeouts; Transports*. [python-httpx.org](https://www.python-httpx.org/)
- Open-Meteo. *Historical Weather API*; *Terms of use*. Data CC BY 4.0. [open-meteo.com/en/docs/historical-weather-api](https://open-meteo.com/en/docs/historical-weather-api)
- GitHub. *Rate limits for the REST API*. [docs.github.com](https://docs.github.com/en/rest/using-the-rest-api/rate-limits-for-the-rest-api)
- scikit-learn developers. *Model persistence*. [scikit-learn.org/stable/model_persistence.html](https://scikit-learn.org/stable/model_persistence.html)
- skops documentation. *Secure persistence with skops*. [skops.readthedocs.io](https://skops.readthedocs.io/)
- FastAPI documentation: [Tutorial – User Guide](https://fastapi.tiangolo.com/tutorial/), [Testing](https://fastapi.tiangolo.com/tutorial/testing/) and [FastAPI in Containers](https://fastapi.tiangolo.com/deployment/docker/) (MIT licence).
- Docker Inc. *Docker: Get started*. [docs.docker.com/get-started](https://docs.docker.com/get-started/)
- GitHub. *Publishing Docker images with GitHub Actions*. [docs.github.com](https://docs.github.com/en/actions/publishing-packages/publishing-docker-images)
- Wiggins, A. (2017). *The Twelve-Factor App*. [12factor.net](https://12factor.net/)
- Breck, E., Cai, S., Nielsen, E., Salib, M. and Sculley, D. (2017). The ML Test Score: a rubric for ML production readiness and technical debt reduction. *IEEE Big Data 2017*. [research.google](https://research.google/pubs/the-ml-test-score-a-rubric-for-ml-production-readiness-and-technical-debt-reduction/)
- Ribeiro, M. T., Wu, T., Guestrin, C. and Singh, S. (2020). Beyond accuracy: behavioral testing of NLP models with CheckList. *ACL 2020*. [arXiv:2005.04118](https://arxiv.org/abs/2005.04118)
- Mohandas, G. *Made With ML* (testing, CI/CD and serving of one project, MIT licence). [madewithml.com](https://madewithml.com/)
- Kästner, C. *Machine Learning in Production: From Models to Products*. Open textbook. [mlip-cmu.github.io/book](https://mlip-cmu.github.io/book/)
- Extra: [extras/01-skops-text-classification-model-card.ipynb](extras/01-skops-text-classification-model-card.ipynb) (skops, MIT).

*Course notebook, CC BY 4.0. Weather data: Open-Meteo, CC BY 4.0. Customs decisions: European Commission, EBTI database (reuse with acknowledgement, Decision 2011/833/EU).*